# Shelf-Life Prediction V2 — Rebuilt for Kaggle

## Data sources (all verified against primary sources, not just cited)

1. **IoT Cold Storage Spoilage Dataset** (Mendeley, `czz68d9fwj`) — real temp/humidity/light/CO2 -> spoilage label, 10,996 rows.
   Download: https://data.mendeley.com/datasets/czz68d9fwj/1 -> "Download All"
2. **USDA FoodKeeper** — real base shelf-life reference per food, auto-downloaded below, no manual step needed.
   Source: https://www.foodsafety.gov/keep-food-safe/foodkeeper-app (data pulled from USDA's public API)
3. **Real-Time Shelf-Life Estimation Model** (Mendeley, `kphtgxn3ff`) — used as a manual literature reference to sanity-check decay constants, not auto-parsed.
   https://data.mendeley.com/datasets/kphtgxn3ff/4 — Abougharib & Awad, published in IEEE Access.

**What's new in V2** (based on review feedback):
- Cross-validation (5-fold), not just a single train/test split
- Hyperparameter tuning via RandomizedSearchCV
- Expanded to 20 test cases instead of 4
- Same schema-safety fix as before (explicit column reordering before every prediction, so a mismatch can't silently happen again)

**Before running:** upload the IoT dataset CSV (from the Mendeley link above) and the FoodKeeper JSON as Kaggle "Add Input" datasets, then update the two paths in the next cell to match.

In [8]:
import os

print("Kaggle Input folders:")
for folder in os.listdir("/kaggle/input"):
    print(f"\n📁 {folder}")
    folder_path = os.path.join("/kaggle/input", folder)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            print(os.path.join(root, file))

Kaggle Input folders:

📁 datasets
/kaggle/input/datasets/parvezthabarak/iot-dataset/A Multi-Parameter Dataset for Machine Learning Bas/Dataset.csv
/kaggle/input/datasets/parvezthabarak/foodkeeper/foodkeeper.json


In [9]:
!pip install -q lightgbm
!pip install -q lightgbm

import json
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

# UPDATE these two paths to match your actual Kaggle inputs
IOT_DATASET_PATH = "/kaggle/input/datasets/parvezthabarak/iot-dataset/A Multi-Parameter Dataset for Machine Learning Bas/Dataset.csv"
FOODKEEPER_PATH = "/kaggle/input/datasets/parvezthabarak/foodkeeper/foodkeeper.json"

print("Update the two paths above if your Kaggle input folder names differ.")

Update the two paths above if your Kaggle input folder names differ.


## Step 1 — Load the real IoT cold-storage dataset

In [10]:
import os
import pandas as pd

if os.path.exists(IOT_DATASET_PATH):
    iot_df = pd.read_csv(IOT_DATASET_PATH)

    # Standardize class labels
    if "Class" in iot_df.columns:
        iot_df["Class"] = iot_df["Class"].str.strip().str.title()

    print("=" * 60)
    print("✅ Loaded real IoT dataset successfully")
    print(f"Rows    : {len(iot_df)}")
    print(f"Columns : {len(iot_df.columns)}")
    print("=" * 60)

    print("\nFirst 5 rows:")
    print(iot_df.head())

    print("\nDataset Shape:")
    print(iot_df.shape)

    print("\nColumn Names:")
    print(list(iot_df.columns))

    if "Class" in iot_df.columns:
        print("\nCorrected Class Distribution:")
        print(iot_df["Class"].value_counts())

else:
    print(f"❌ File not found:\n{IOT_DATASET_PATH}")
    iot_df = None

✅ Loaded real IoT dataset successfully
Rows    : 10995
Columns : 6

First 5 rows:
       Fruit  Temp  Humid (%)  Light (Fux)  CO2 (pmm) Class
0     Orange    22         95     7.358649        361  Good
1     Orange    24         95    14.835898        370   Bad
2  Pineapple    25         95    10.104045        355   Bad
3     Banana    25         89    20.179643        388  Good
4     Tomato    23         90    12.621448        316  Good

Dataset Shape:
(10995, 6)

Column Names:
['Fruit', 'Temp', 'Humid (%)', 'Light (Fux)', 'CO2 (pmm)', 'Class']

Corrected Class Distribution:
Class
Good    5667
Bad     5328
Name: count, dtype: int64


## Step 2 — Sanity-check the physical relationship using real data

In [11]:
if iot_df is not None:

    # Use the verified column names from the IoT dataset
    temp_col = "Temp"
    humid_col = "Humid (%)"
    class_col = "Class"

    print("=" * 60)
    print("IoT DATASET ANALYSIS")
    print("=" * 60)

    print("\nMean Temperature by Spoilage Class")
    print("-" * 60)
    print(iot_df.groupby(class_col)[temp_col].mean().round(2))

    print("\nMean Humidity by Spoilage Class")
    print("-" * 60)
    print(iot_df.groupby(class_col)[humid_col].mean().round(2))

    print("\nInterpretation")
    print("-" * 60)

    temp_means = iot_df.groupby(class_col)[temp_col].mean()

    if "Good" in temp_means.index and "Bad" in temp_means.index:
        if temp_means["Bad"] > temp_means["Good"]:
            print("✓ Bad samples have a higher average temperature than Good samples.")
            print("✓ This supports the temperature–spoilage relationship used")
            print("  in the synthetic shelf-life generator.")
        else:
            print("⚠ Bad samples do not show a higher average temperature.")
            print("  Review the assumptions before using this relationship.")

else:
    print("Skipping IoT analysis - dataset not loaded.")

IoT DATASET ANALYSIS

Mean Temperature by Spoilage Class
------------------------------------------------------------
Class
Bad     24.33
Good    23.38
Name: Temp, dtype: float64

Mean Humidity by Spoilage Class
------------------------------------------------------------
Class
Bad     94.97
Good    92.17
Name: Humid (%), dtype: float64

Interpretation
------------------------------------------------------------
✓ Bad samples have a higher average temperature than Good samples.
✓ This supports the temperature–spoilage relationship used
  in the synthetic shelf-life generator.


## Step 3 — Load USDA FoodKeeper and build the base shelf-life table (23 produce types)

In [17]:
# Step 3 — Load USDA FoodKeeper and build the base shelf-life table (25 produce types)

import re

products = None

try:
    with open(FOODKEEPER_PATH, "r", encoding="utf-8") as f:
        foodkeeper = json.load(f)

    print("✅ FoodKeeper loaded successfully.")

    product_sheet = next(s for s in foodkeeper["sheets"] if s["name"].lower() == "product")
    category_sheet = next(s for s in foodkeeper["sheets"] if s["name"].lower() == "category")

    products = []
    for row in product_sheet["data"]:
        item = {}
        for part in row:
            item.update(part)
        products.append(item)

    categories = {}
    for row in category_sheet["data"]:
        item = {}
        for part in row:
            item.update(part)
        categories[item["ID"]] = item["Subcategory_Name"] or item["Category_Name"]

    print(f"Loaded {len(products)} FoodKeeper products.")

except Exception as e:
    print(f"⚠️ Could not load FoodKeeper ({e})")
    print("Using literature fallback values where needed.")
    products = None
    categories = {}


# Restrict matching to the real "Produce" categories, so juices/sauces/
# pies/nectars (which share keywords with fresh produce) never leak in.
PRODUCE_CATEGORY_IDS = {
    cid for cid, name in categories.items()
    if name in ("Fresh Fruits", "Fresh Vegetables")
}

# ------------------------------------------------------------
# Full classifier produce list (25 classes)
# ------------------------------------------------------------
TARGET_ITEMS = [
    "Apple", "Banana", "Bellpepper", "Bitter_Gourd", "Carrot",
    "Cucumber", "Grape", "Grapes", "Guava", "Jujube",
    "Kaki", "Lemon", "Lime", "Lulo", "Mango",
    "Orange", "Papaya", "Peach", "Pear", "Pomegranate",
    "Potato", "Strawberry", "Tamarillo", "Tomato",
    "Watermelon",
]


def unit_to_days(value, unit):
    if value is None or unit is None:
        return None
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    unit = unit.lower()
    if "day" in unit:
        return value
    if "week" in unit:
        return value * 7
    if "month" in unit:
        return value * 30
    if "year" in unit:
        return value * 365
    return None


def days_from_item(item):
    """Refrigerator fields ONLY. This project models refrigerated storage,
    so Pantry values (often 'When Ripe' or a very different number) must
    never leak in as a refrigerated shelf life."""
    fields = [
        ("Refrigerate_Min", "Refrigerate_Metric"),
        ("DOP_Refrigerate_Min", "DOP_Refrigerate_Metric"),
        ("Refrigerate_Max", "Refrigerate_Metric"),
        ("DOP_Refrigerate_Max", "DOP_Refrigerate_Metric"),
    ]
    for value_field, metric_field in fields:
        days = unit_to_days(item.get(value_field), item.get(metric_field))
        if days is not None:
            return days
    return None


FALLBACK_DAYS = {
    "Apple": 25, "Banana": 6, "Bellpepper": 14, "Bitter_Gourd": 10, "Carrot": 24,
    "Cucumber": 7, "Grape": 10, "Grapes": 10, "Guava": 7, "Jujube": 21,
    "Kaki": 30, "Lemon": 25, "Lime": 25, "Lulo": 20, "Mango": 6,
    "Orange": 25, "Papaya": 3, "Peach": 4, "Pear": 6, "Pomegranate": 45,
    "Potato": 60, "Strawberry": 5, "Tamarillo": 10, "Tomato": 7, "Watermelon": 14,
}

FOODKEEPER_ALIASES = {
    "Apple": ["apple", "apples"],
    "Banana": ["banana", "bananas"],
    "Bellpepper": ["bell pepper", "bell peppers", "pepper", "peppers"],
    "Carrot": ["carrot", "carrots"],
    "Cucumber": ["cucumber", "cucumbers"],
    "Grape": ["grape", "grapes"],
    "Grapes": ["grape", "grapes"],
    "Guava": ["guava"],
    "Lemon": ["lemon", "lemons"],
    "Lime": ["lime", "limes"],
    "Mango": ["mango", "mangoes"],
    "Orange": ["orange", "oranges"],
    "Papaya": ["papaya"],
    "Peach": ["peach", "peaches"],
    "Pear": ["pear", "pears"],
    "Pomegranate": ["pomegranate"],
    "Potato": ["potato", "potatoes"],
    "Strawberry": ["strawberry", "strawberries"],
    "Tomato": ["tomato", "tomatoes"],
    "Watermelon": ["watermelon"],
}

# Secondary safety net even within the Produce categories (e.g. "Baby carrots")
IGNORE_WORDS = ["baby", "pickled", "canned", "dried", "frozen"]


def clean(text):
    text = (text or "").lower()
    text = text.replace("_", " ")
    text = text.replace(",", " ")
    text = " ".join(text.split())
    return text


def word_match(alias, text):
    # Whole-word match, e.g. "lemon" must NOT match inside "lemongrass"
    return re.search(rf"\b{re.escape(alias)}\b", text) is not None


foodkeeper_days = {}
foodkeeper_matches = {}

if products is not None:

    for target, raw_aliases in FOODKEEPER_ALIASES.items():

        aliases = [clean(a) for a in raw_aliases]
        candidates = []  # (days, item, is_exact_name_match)

        for item in products:
            if item.get("Category_ID") not in PRODUCE_CATEGORY_IDS:
                continue

            name_raw = (item.get("Name") or "").lower()
            if any(word in name_raw for word in IGNORE_WORDS):
                continue

            name = clean(item.get("Name", ""))
            search_text = clean(
                (item.get("Name", "") or "")
                + " " + str(item.get("Name_subtitle", "") or "")
                + " " + str(item.get("Keywords", "") or "")
            )

            is_exact = name in aliases
            is_keyword = any(word_match(alias, search_text) for alias in aliases)
            if not (is_exact or is_keyword):
                continue

            days = days_from_item(item)
            if days is None:
                continue

            candidates.append((days, item, is_exact))

        if candidates:
            # Prefer an exact-name row over a keyword/combo row; among ties,
            # take the shorter (more conservative) value.
            exact_candidates = [c for c in candidates if c[2]]
            pool = exact_candidates if exact_candidates else candidates
            best_days, best_item, _ = min(pool, key=lambda c: c[0])

            foodkeeper_days[target] = best_days
            foodkeeper_matches[target] = best_item

            print(
                f"{target:14s} <- matched '{best_item['Name']}' -> "
                f"{best_days:.1f} days (FoodKeeper)"
            )

for item in TARGET_ITEMS:
    if item not in foodkeeper_days:
        foodkeeper_days[item] = FALLBACK_DAYS[item]
        print(
            f"{item:14s} <- no FoodKeeper match, "
            f"using literature fallback -> {FALLBACK_DAYS[item]} days"
        )

print("\n" + "=" * 70)
print(f"Final base_days table ({len(TARGET_ITEMS)} classes)")
print("=" * 70)
print(foodkeeper_days)

✅ FoodKeeper loaded successfully.
Loaded 661 FoodKeeper products.
Apple          <- matched 'Apples' -> 28.0 days (FoodKeeper)
Banana         <- matched 'Bananas' -> 3.0 days (FoodKeeper)
Bellpepper     <- matched 'Peppers' -> 4.0 days (FoodKeeper)
Carrot         <- matched 'Carrots, parsnips' -> 14.0 days (FoodKeeper)
Cucumber       <- matched 'Cucumbers' -> 4.0 days (FoodKeeper)
Grape          <- matched 'Grapes' -> 7.0 days (FoodKeeper)
Grapes         <- matched 'Grapes' -> 7.0 days (FoodKeeper)
Guava          <- matched 'Guava' -> 2.0 days (FoodKeeper)
Lemon          <- matched 'Citrus fruit' -> 10.0 days (FoodKeeper)
Lime           <- matched 'Citrus fruit' -> 10.0 days (FoodKeeper)
Mango          <- matched 'Papaya, mango, feijoa, passionfruit, casaha melon' -> 7.0 days (FoodKeeper)
Orange         <- matched 'Citrus fruit' -> 10.0 days (FoodKeeper)
Papaya         <- matched 'Papaya, mango, feijoa, passionfruit, casaha melon' -> 7.0 days (FoodKeeper)
Peach          <- matched 'Pea

## Step 4 — Synthetic layer for the confirmed real gap (packaging, freshness curve)

In [23]:
# Step 4 — Synthetic layer for the confirmed real gap (packaging, freshness curve)

rng = np.random.default_rng(42)

RH_OPT = {
    "Apple": (90, 12),
    "Banana": (90, 10),
    "Bellpepper": (90, 10),
    "Bitter_Gourd": (90, 8),
    "Carrot": (95, 10),
    "Cucumber": (90, 10),
    "Grape": (92, 8),
    "Grapes": (92, 8),
    "Guava": (88, 10),
    "Jujube": (90, 10),
    "Kaki": (90, 10),
    "Lemon": (88, 14),
    "Lime": (88, 14),
    "Lulo": (90, 10),
    "Mango": (90, 10),
    "Orange": (90, 14),
    "Papaya": (88, 10),
    "Peach": (90, 8),
    "Pear": (90, 12),
    "Pomegranate": (85, 12),
    "Potato": (85, 15),
    "Strawberry": (90, 8),
    "Tamarillo": (90, 10),
    "Tomato": (90, 10),
    "Watermelon": (88, 12),
}

CLIMACTERIC = {
    "Apple": True,
    "Banana": True,
    "Bellpepper": False,
    "Bitter_Gourd": False,
    "Carrot": False,
    "Cucumber": False,
    "Grape": False,
    "Grapes": False,
    "Guava": True,
    "Jujube": True,
    "Kaki": True,
    "Lemon": False,
    "Lime": False,
    "Lulo": True,
    "Mango": True,
    "Orange": False,
    "Papaya": True,
    "Peach": True,
    "Pear": True,
    "Pomegranate": False,
    "Potato": False,
    "Strawberry": False,
    "Tamarillo": True,
    "Tomato": True,
    "Watermelon": False,
}

# (Mean Temp, Temp Std, Mean RH, RH Std)
STORAGE_AREAS = {
    "fridge": (4.0, 1.5, 88, 5),
    "pantry": (20.0, 3.0, 55, 10),
    "counter": (24.0, 4.0, 45, 12),
}

PACKAGING_TYPES = [
    "unpackaged",
    "plastic_wrap",
    "perforated_bag",
    "sealed_container",
]


def packaging_factor(pkg, climacteric):

    base = {
        "unpackaged": 1.00,
        "plastic_wrap": 1.15,
        "perforated_bag": 1.25,
        "sealed_container": 1.35,
    }[pkg]

    # Climacteric fruits deteriorate faster if sealed
    if climacteric and pkg == "sealed_container":
        base *= 0.75

    return base


Q10 = 2.5
T_REF = 4.0
FRESHNESS_THRESHOLD = 40.0


def simulate_row(item):

    base_days = foodkeeper_days[item]

    rh_opt, rh_sigma = RH_OPT[item]

    climacteric = CLIMACTERIC[item]

    # More realistic storage probabilities
    storage_area = rng.choice(
        list(STORAGE_AREAS.keys()),
        p=[0.60, 0.20, 0.20],
    )

    t_mean, t_std, rh_mean, rh_std = STORAGE_AREAS[storage_area]
    # Generate realistic storage temperature (0–40°C)
    temperature = np.clip(
    rng.normal(t_mean, t_std),
    0.0,
    40.0,
    )

    humidity = np.clip(
        rng.normal(rh_mean, rh_std),
        20,
        100,
    )

    packaging = rng.choice(PACKAGING_TYPES)

    # More realistic freshness distribution
    freshness_pct = rng.triangular(
        45,
        85,
        100,
    )

    temp_rate = Q10 ** ((temperature - T_REF) / 10.0)

    humidity_factor = max(
        np.exp(
            -((humidity - rh_opt) ** 2)
            / (2 * rh_sigma ** 2)
        ),
        0.15,
    )

    pack_factor = packaging_factor(
        packaging,
        climacteric,
    )

    k_base = np.log(
        100.0 / FRESHNESS_THRESHOLD
    ) / base_days

    k_eff = (
        k_base
        * temp_rate
        / (humidity_factor * pack_factor)
    )

    remaining_days = np.log(
        freshness_pct / FRESHNESS_THRESHOLD
    ) / k_eff

    remaining_days = max(
        remaining_days,
        0.0,
    )

    # Controlled natural variation
    noise = np.clip(
        rng.normal(1.0, 0.15),
        0.70,
        1.30,
    )

    remaining_days *= noise

    remaining_days = round(
        max(remaining_days, 0.0),
        2,
    )

    return {
        "produce_type": item,
        "temperature_c": round(temperature, 1),
        "humidity_pct": round(humidity, 1),
        "storage_area": storage_area,
        "packaging_material": packaging,
        "freshness_pct": round(freshness_pct, 1),
        "remaining_shelf_life_days": remaining_days,
    }


def generate_dataset(n_per_item=400):

    rows = []

    for item in TARGET_ITEMS:
        for _ in range(n_per_item):
            rows.append(simulate_row(item))

    return pd.DataFrame(rows)


df = generate_dataset(n_per_item=400)

CSV_LOCAL = f"{WORK_DIR}/synthetic_shelf_life.csv"

df.to_csv(
    CSV_LOCAL,
    index=False,
)

print("=" * 70)
print(
    f"Generated {len(df):,} synthetic samples across "
    f"{df['produce_type'].nunique()} produce types."
)
print("=" * 70)

print("\nSamples per produce type:")
print(df["produce_type"].value_counts().sort_index())

print("\nDataset Preview:")
display(df.head())

Generated 10,000 synthetic samples across 25 produce types.

Samples per produce type:
produce_type
Apple           400
Banana          400
Bellpepper      400
Bitter_Gourd    400
Carrot          400
Cucumber        400
Grape           400
Grapes          400
Guava           400
Jujube          400
Kaki            400
Lemon           400
Lime            400
Lulo            400
Mango           400
Orange          400
Papaya          400
Peach           400
Pear            400
Pomegranate     400
Potato          400
Strawberry      400
Tamarillo       400
Tomato          400
Watermelon      400
Name: count, dtype: int64

Dataset Preview:


,produce_type,temperature_c,humidity_pct,storage_area,packaging_material,freshness_pct,remaining_shelf_life_days
0,Apple,16.9,62.5,pantry,unpackaged,59.4,0.45
1,Apple,19.1,54.8,pantry,perforated_bag,76.5,1.06
2,Apple,24.3,58.5,counter,perforated_bag,67.4,0.49
3,Apple,5.3,87.8,fridge,plastic_wrap,85.9,20.99
4,Apple,23.4,39.9,counter,sealed_container,77.0,0.54


In [24]:
print(df["temperature_c"].min())
print(df["temperature_c"].max())

0.0
36.2


## Step 5 — Cross-validation (new)

A single train/test split gives one number; 5-fold cross-validation gives a
distribution, which is a stronger answer to "how did you validate this?"
**Important honest caveat carried over from before:** since the whole dataset
comes from one formula-driven generator, cross-validation will likely also
show a high, consistent score — that confirms the model fits the *modeled
relationship* reliably, not that it's been validated against real-world
ground-truth shelf-life measurements. Report it as such.

In [26]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error

cat_cols = [
    "produce_type",
    "storage_area",
    "packaging_material",
]

for c in cat_cols:
    df[c] = df[c].astype("category")

X = df.drop(columns=["remaining_shelf_life_days"])
y = df["remaining_shelf_life_days"]

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

r2_scores = []
mae_scores = []

print("=" * 70)
print("5-FOLD CROSS VALIDATION")
print("=" * 70)

for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = lgb.LGBMRegressor(
        n_estimators=150,
        num_leaves=15,
        max_depth=5,
        learning_rate=0.08,
        min_child_samples=10,
        verbose=-1,
    )

    model.fit(
        X_train,
        y_train,
        categorical_feature=cat_cols,
    )

    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)

    r2_scores.append(r2)
    mae_scores.append(mae)

    print(
        f"Fold {fold}: "
        f"R² = {r2:.4f} | "
        f"MAE = {mae:.2f} days"
    )

print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Mean R²  : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}"
)

print(
    f"Mean MAE : {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f} days"
)

print("\nNote:")
print("These scores evaluate how well LightGBM learns the")
print("synthetic shelf-life model. They do NOT represent")
print("validation against real-world measured shelf-life.")

print("\nFold-wise R²:", [round(x,4) for x in r2_scores])
print("Fold-wise MAE:", [round(x,2) for x in mae_scores])

5-FOLD CROSS VALIDATION
Fold 1: R² = 0.9589 | MAE = 0.67 days
Fold 2: R² = 0.9583 | MAE = 0.64 days
Fold 3: R² = 0.9587 | MAE = 0.65 days
Fold 4: R² = 0.9588 | MAE = 0.62 days
Fold 5: R² = 0.9476 | MAE = 0.67 days

CROSS-VALIDATION SUMMARY
Mean R²  : 0.9565 ± 0.0044
Mean MAE : 0.65 ± 0.02 days

Note:
These scores evaluate how well LightGBM learns the
synthetic shelf-life model. They do NOT represent
validation against real-world measured shelf-life.

Fold-wise R²: [0.9589, 0.9583, 0.9587, 0.9588, 0.9476]
Fold-wise MAE: [0.67, 0.64, 0.65, 0.62, 0.67]


In [27]:
cv_results = {
    "mean_r2": float(np.mean(r2_scores)),
    "std_r2": float(np.std(r2_scores)),
    "mean_mae": float(np.mean(mae_scores)),
    "std_mae": float(np.std(mae_scores)),
}

print(cv_results)

{'mean_r2': 0.9564762946795453, 'std_r2': 0.0044250352975120566, 'mean_mae': 0.6497895341109927, 'std_mae': 0.019721101721853224}


## Step 6 — Hyperparameter tuning (new)

Searches a reasonable range of LightGBM parameters. The ceiling for
improvement here is naturally small since the baseline already fits the
synthetic relationship well — don't expect a dramatic jump, this is about
being able to say tuning was actually tried, not about chasing a number.

In [28]:
# ==============================================================
# Step 6 — Hyperparameter Tuning
# ==============================================================

print("=" * 70)
print("HYPERPARAMETER TUNING")
print("=" * 70)

# Baseline parameters (used in Step 5)
BASELINE_PARAMS = {
    "num_leaves": 15,
    "learning_rate": 0.08,
    "max_depth": 5,
    "min_child_samples": 10,
}

# Search space
param_dist = {
    "num_leaves": [15, 20, 31],
    "learning_rate": [0.05, 0.08, 0.10, 0.15],
    "max_depth": [4, 5, 6, 8],
    "min_child_samples": [3, 5, 10],
}

search = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(
        n_estimators=150,
        verbose=-1,
    ),
    param_distributions=param_dist,
    n_iter=30,
    scoring="r2",
    cv=kfold,
    random_state=42,
    n_jobs=-1,
)

# Train the search
search.fit(
    X,
    y,
    categorical_feature=cat_cols,
)

# --------------------------------------------------------------
# Compare tuned model with baseline
# --------------------------------------------------------------
baseline_r2 = np.mean(r2_scores)
tuned_r2 = search.best_score_

print("\nBest Hyperparameters")
print("-" * 70)

for key, value in search.best_params_.items():
    print(f"{key:20s}: {value}")

print("\nPerformance Comparison")
print("-" * 70)

print(f"Baseline CV R² : {baseline_r2:.4f}")
print(f"Tuned CV R²    : {tuned_r2:.4f}")
print(f"Improvement    : {tuned_r2 - baseline_r2:+.4f}")

# --------------------------------------------------------------
# Safety fallback
# --------------------------------------------------------------
if tuned_r2 >= baseline_r2:

    best_params = search.best_params_.copy()

    print("\n✅ Using tuned hyperparameters.")

else:

    best_params = BASELINE_PARAMS.copy()

    print("\n⚠ Tuned model did not outperform the baseline.")
    print("⚠ Falling back to baseline hyperparameters.")

print("\nFinal Parameters Used")
print("-" * 70)

for key, value in best_params.items():
    print(f"{key:20s}: {value}")

HYPERPARAMETER TUNING

Best Hyperparameters
----------------------------------------------------------------------
num_leaves          : 15
min_child_samples   : 10
max_depth           : 6
learning_rate       : 0.15

Performance Comparison
----------------------------------------------------------------------
Baseline CV R² : 0.9565
Tuned CV R²    : 0.9583
Improvement    : +0.0018

✅ Using tuned hyperparameters.

Final Parameters Used
----------------------------------------------------------------------
num_leaves          : 15
min_child_samples   : 10
max_depth           : 6
learning_rate       : 0.15


## Step 7 — Train the final model with the best parameters found

In [31]:
# ==============================================================
# Step 7 — Train Final Model and Evaluate
# ==============================================================

import json
import os

print("=" * 70)
print("FINAL MODEL TRAINING")
print("=" * 70)

# --------------------------------------------------------------
# Train/Test Split
# --------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
)

# --------------------------------------------------------------
# Train Final LightGBM Model
# --------------------------------------------------------------
model = lgb.LGBMRegressor(
    n_estimators=150,
    verbose=-1,
    **best_params,
)

model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols,
)

# --------------------------------------------------------------
# Predictions
# --------------------------------------------------------------
preds = model.predict(X_test)

# --------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------
test_mae = mean_absolute_error(y_test, preds)
test_r2 = r2_score(y_test, preds)

print("\nHeld-Out Test Performance")
print("-" * 70)
print(f"Test MAE : {test_mae:.2f} days")
print(f"Test R²  : {test_r2:.4f}")

print("\nModel Information")
print("-" * 70)
print(f"Training Samples : {len(X_train):,}")
print(f"Testing Samples  : {len(X_test):,}")
print(f"Features         : {X.shape[1]}")

print("\nImportant Note")
print("-" * 70)
print(
    "These metrics evaluate how accurately the model learns the\n"
    "synthetic shelf-life relationship generated from FoodKeeper,\n"
    "IoT-inspired environmental conditions, and literature-based\n"
    "decay assumptions. They do NOT represent validation against\n"
    "real-world measured shelf-life observations."
)

# --------------------------------------------------------------
# Save Model
# --------------------------------------------------------------
MODEL_LOCAL = f"{WORK_DIR}/shelf_life_model.txt"

model.booster_.save_model(MODEL_LOCAL)

# --------------------------------------------------------------
# Save Evaluation Results
# --------------------------------------------------------------
evaluation_results = {
    "test_r2": float(test_r2),
    "test_mae_days": float(test_mae),
    "training_samples": int(len(X_train)),
    "testing_samples": int(len(X_test)),
    "best_parameters": best_params,
}

with open(f"{WORK_DIR}/evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("Evaluation results saved.")

# --------------------------------------------------------------
# Save Feature Metadata (Recommended)
# --------------------------------------------------------------
feature_metadata = {
    "feature_names": list(X.columns),
    "categorical_features": cat_cols,
}

with open(f"{WORK_DIR}/feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=4)

print("Feature metadata saved.")

# --------------------------------------------------------------
# Final Summary
# --------------------------------------------------------------
print("\nModel Saved Successfully")
print("-" * 70)
print(f"Model File           : {MODEL_LOCAL}")
print(f"Model Size           : {os.path.getsize(MODEL_LOCAL)/1024:.1f} KB")
print(f"Evaluation File      : {WORK_DIR}/evaluation_results.json")
print(f"Feature Metadata     : {WORK_DIR}/feature_metadata.json")

print("\nEvaluation Summary")
print("-" * 70)
print(json.dumps(evaluation_results, indent=4))

FINAL MODEL TRAINING

Held-Out Test Performance
----------------------------------------------------------------------
Test MAE : 0.64 days
Test R²  : 0.9598

Model Information
----------------------------------------------------------------------
Training Samples : 8,500
Testing Samples  : 1,500
Features         : 6

Important Note
----------------------------------------------------------------------
These metrics evaluate how accurately the model learns the
synthetic shelf-life relationship generated from FoodKeeper,
IoT-inspired environmental conditions, and literature-based
decay assumptions. They do NOT represent validation against
real-world measured shelf-life observations.
Evaluation results saved.
Feature metadata saved.

Model Saved Successfully
----------------------------------------------------------------------
Model File           : /kaggle/working/shelf_life_model.txt
Model Size           : 220.1 KB
Evaluation File      : /kaggle/working/evaluation_results.json
Feature

## Step 8 — Load the saved model and test on 20 produce items (expanded from 4)

In [33]:
# ==============================================================
# Step 8 — Load Saved Model and Run Sample Predictions
# ==============================================================

print("=" * 70)
print("MODEL INFERENCE")
print("=" * 70)

# --------------------------------------------------------------
# Load trained model
# --------------------------------------------------------------
loaded_model = lgb.Booster(model_file=MODEL_LOCAL)

# --------------------------------------------------------------
# Sample test data (all 25 produce classes)
# --------------------------------------------------------------
test_data = pd.DataFrame({

    "produce_type": [
        "Apple","Banana","Bellpepper","Bitter_Gourd","Carrot",
        "Cucumber","Grape","Grapes","Guava","Jujube",
        "Kaki","Lemon","Lime","Lulo","Mango",
        "Orange","Papaya","Peach","Pear","Pomegranate",
        "Potato","Strawberry","Tamarillo","Tomato","Watermelon"
    ],

    "temperature_c": [
        4,5,4,20,4,
        8,4,4,20,20,
        4,5,20,4,20,
        4,20,4,4,4,
        20,4,20,5,8
    ],

    "humidity_pct": [
        90,85,90,60,95,
        90,92,92,55,45,
        90,85,45,90,55,
        90,55,90,90,85,
        55,90,55,90,88
    ],

    "storage_area": [
        "fridge","fridge","fridge","counter","fridge",
        "fridge","fridge","fridge","counter","counter",
        "fridge","fridge","counter","fridge","counter",
        "fridge","counter","fridge","fridge","fridge",
        "counter","fridge","counter","fridge","fridge"
    ],

    "packaging_material": [
        "plastic_wrap","unpackaged","perforated_bag","unpackaged","perforated_bag",
        "plastic_wrap","unpackaged","unpackaged","unpackaged","unpackaged",
        "unpackaged","unpackaged","unpackaged","unpackaged","unpackaged",
        "unpackaged","unpackaged","unpackaged","plastic_wrap","unpackaged",
        "unpackaged","plastic_wrap","unpackaged","plastic_wrap","plastic_wrap"
    ],

    "freshness_pct": [
        85,70,90,80,95,
        80,85,85,75,80,
        90,88,75,80,70,
        85,65,75,85,90,
        80,90,80,85,85
    ],
})

# --------------------------------------------------------------
# Convert categorical columns
# --------------------------------------------------------------
for col in cat_cols:
    test_data[col] = test_data[col].astype("category")

# --------------------------------------------------------------
# Ensure feature order matches training
# --------------------------------------------------------------
test_data = test_data[loaded_model.feature_name()]

# --------------------------------------------------------------
# Predict
# --------------------------------------------------------------
predictions = loaded_model.predict(test_data)

# Shelf life can never be negative
predictions = np.clip(predictions, 0, None)

# --------------------------------------------------------------
# Build results table
# --------------------------------------------------------------
results = test_data.copy()

results["remaining_shelf_life_days"] = np.round(predictions, 2)

results["remaining_shelf_life_hours"] = np.round(
    predictions * 24,
    1,
)

results = results.round({
    "temperature_c": 1,
    "humidity_pct": 1,
    "freshness_pct": 1,
})

results = results.sort_values(
    "remaining_shelf_life_days",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------------------
# Display results
# --------------------------------------------------------------
print("\nPrediction Results")
print("-" * 70)

display(results)

# --------------------------------------------------------------
# Save predictions
# --------------------------------------------------------------
PREDICTION_FILE = f"{WORK_DIR}/sample_predictions.csv"

results.to_csv(
    PREDICTION_FILE,
    index=False,
)

print("\nPrediction file saved successfully.")
print(f"Location : {PREDICTION_FILE}")
print(f"Rows     : {len(results)}")
print(f"Features : {len(results.columns)}")

MODEL INFERENCE

Prediction Results
----------------------------------------------------------------------


,produce_type,temperature_c,humidity_pct,storage_area,packaging_material,freshness_pct,remaining_shelf_life_days,remaining_shelf_life_hours
0,Apple,4,90,fridge,plastic_wrap,85,25.05,601.2
1,Kaki,4,90,fridge,unpackaged,90,24.72,593.4
2,Pomegranate,4,85,fridge,unpackaged,90,23.93,574.4
3,Carrot,4,95,fridge,perforated_bag,95,18.03,432.7
4,Lulo,4,90,fridge,unpackaged,80,14.65,351.7
5,Orange,4,90,fridge,unpackaged,85,8.42,202.1
6,Lemon,5,85,fridge,unpackaged,88,8.02,192.6
7,Grape,4,92,fridge,unpackaged,85,5.87,141.0
8,Grapes,4,92,fridge,unpackaged,85,5.86,140.8
9,Bellpepper,4,90,fridge,perforated_bag,90,4.59,110.1



Prediction file saved successfully.
Location : /kaggle/working/sample_predictions.csv
Rows     : 25
Features : 8


## Step 9 — Feature importance

In [34]:
# ==============================================================
# Step 9 — Feature Importance Analysis
# ==============================================================

print("=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

# --------------------------------------------------------------
# Get feature importance
# --------------------------------------------------------------
importance = pd.DataFrame({
    "feature": loaded_model.feature_name(),
    "importance": loaded_model.feature_importance(importance_type="gain")
})

importance = importance.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------------------
# Display table
# --------------------------------------------------------------
print("\nFeature Importance Ranking")
print("-" * 70)

display(importance)

# --------------------------------------------------------------
# Save feature importance
# --------------------------------------------------------------
IMPORTANCE_FILE = f"{WORK_DIR}/feature_importance.csv"

importance.to_csv(
    IMPORTANCE_FILE,
    index=False,
)

print("\nFeature importance saved successfully.")
print(f"Location : {IMPORTANCE_FILE}")

# --------------------------------------------------------------
# Simple text summary
# --------------------------------------------------------------
print("\nMost Important Feature")
print("-" * 70)

print(
    f"{importance.iloc[0]['feature']} "
    f"(Importance = {importance.iloc[0]['importance']:.2f})"
)

FEATURE IMPORTANCE

Feature Importance Ranking
----------------------------------------------------------------------


,feature,importance
0,produce_type,644063.977508
1,temperature_c,256347.539974
2,humidity_pct,200542.612878
3,freshness_pct,119914.462035
4,packaging_material,10847.916989
5,storage_area,558.118538



Feature importance saved successfully.
Location : /kaggle/working/feature_importance.csv

Most Important Feature
----------------------------------------------------------------------
produce_type (Importance = 644063.98)


## Step 10 — Save everything

In [35]:
import os

print("=" * 70)
print("DIRECTORY TREE")
print("=" * 70)

for root, dirs, files in os.walk(WORK_DIR):
    level = root.replace(WORK_DIR, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{indent}    {f} ({size:.1f} KB)")

DIRECTORY TREE
working/
    synthetic_shelf_life.csv (470.8 KB)
    sample_predictions.csv (1.2 KB)
    evaluation_results.json (0.3 KB)
    feature_metadata.json (0.3 KB)
    shelf_life_model.txt (220.1 KB)
    feature_importance.csv (0.2 KB)
    .virtual_documents/
        __notebook_source__.ipynb (27.8 KB)


In [36]:
import os
import shutil
import json

print("=" * 70)
print("SAVING FINAL PROJECT FILES")
print("=" * 70)

FINAL_DIR = f"{WORK_DIR}/shelf_life_final"
os.makedirs(FINAL_DIR, exist_ok=True)

# Copy generated files
files = [
    "shelf_life_model.txt",
    "synthetic_shelf_life.csv",
    "evaluation_results.json",
    "feature_metadata.json",
    "feature_importance.csv",
    "sample_predictions.csv",
]

for file in files:
    shutil.copy2(
        os.path.join(WORK_DIR, file),
        os.path.join(FINAL_DIR, file),
    )

# Save final hyperparameters actually used
with open(os.path.join(FINAL_DIR, "best_hyperparameters.json"), "w") as f:
    json.dump(best_params, f, indent=4)

print("\nSaved Files")
print("-" * 70)

for file in sorted(os.listdir(FINAL_DIR)):
    print(f"✓ {file}")

print(f"\nTotal Files : {len(os.listdir(FINAL_DIR))}")
print(f"Location    : {FINAL_DIR}")

print("\nProject is ready for download from Kaggle Output.")

SAVING FINAL PROJECT FILES

Saved Files
----------------------------------------------------------------------
✓ best_hyperparameters.json
✓ evaluation_results.json
✓ feature_importance.csv
✓ feature_metadata.json
✓ sample_predictions.csv
✓ shelf_life_model.txt
✓ synthetic_shelf_life.csv

Total Files : 7
Location    : /kaggle/working/shelf_life_final

Project is ready for download from Kaggle Output.


In [38]:
import os
import shutil

WORK_DIR = "/kaggle/working"
FINAL_DIR = os.path.join(WORK_DIR, "shelf_life_final")
ZIP_PATH = os.path.join(WORK_DIR, "shelf_life_final")

# Create ZIP archive
shutil.make_archive(
    base_name=ZIP_PATH,
    format="zip",
    root_dir=FINAL_DIR
)

print("=" * 70)
print("ZIP FILE CREATED")
print("=" * 70)
print(f"Location : {ZIP_PATH}.zip")
print(f"Size     : {os.path.getsize(ZIP_PATH + '.zip') / 1024:.2f} KB")

ZIP FILE CREATED
Location : /kaggle/working/shelf_life_final.zip
Size     : 170.57 KB


## Checklist
- [x] Real IoT dataset used to sanity-check the physical relationship
- [x] Real USDA FoodKeeper used as the base shelf-life reference
- [x] Synthetic layer built from actual postharvest science (Q10 kinetics, climacteric behavior), not arbitrary values
- [x] Cross-validation added (5-fold), not just a single split
- [x] Hyperparameter tuning added (RandomizedSearchCV)
- [x] Test cases expanded from 4 to 20
- [x] Schema-safety fix retained (explicit column reordering before every prediction)
- [x] R² caveat stated explicitly, both in code output and here, for your report

**Honest scope note carried over:** searched for an additional broad real
shelf-life dataset to strengthen this further — found only narrow,
single-item studies (dates specifically, using gas sensors/NIR — different
format, not a clean fit) rather than anything covering many produce types
cleanly. The three sources already in use remain the strongest available
foundation for this task.